# Diagnostic A' — Unify the data pipeline to test the resize-kernel leak

Hypothesis: ConstrainedCNN's in-distribution AUC = 1.000 by epoch 3 is too
good to be true. The training set has a label-correlated confound: real
FFHQ and OOD Stable Diffusion are PIL-rendered, but in-distribution
FaceSwap was downsampled with `cv2.INTER_AREA` and saved with
`cv2.imwrite`. ConstrainedCNN suppresses content and amplifies
high-frequency residuals — exactly the band where INTER_AREA differs
from LANCZOS — so it could be locking onto the resampler fingerprint
instead of any manipulation cue.

This notebook:
1. **Pre-flight A** — bit-identity check. cv2.imread vs PIL.open on the
   existing `data/faceswap/*.png`. If decoders agree, the cv2-vs-PIL
   *encoder* split cannot leak through to the model.
2. **Pre-flight B** — resize-kernel demo. INTER_AREA vs LANCZOS on one
   high-res Kaggle source frame. Establishes that the resize asymmetry
   exists at the pixel level.
3. **Re-render** FaceSwap from the Kaggle source through a PIL pipeline
   into `data_unified/faceswap/`. Original `data/faceswap/` (cv2-rendered)
   is left untouched.
4. **Regenerate splits** at `data_unified/splits/`, pointing at original
   `data/real/`, the unified `data_unified/faceswap/`, and original
   `data/diffusion_sd15/`. Single-variable change.
5. **Retrain** in `02_train_models.ipynb` after flipping `SPLITS_DIR` to
   `data_unified/splits/`. If in-dist AUC drops from 1.000, the resize
   asymmetry was the leak.

## 0. Colab setup

Same idempotent pattern as notebook 02: clone repo if missing, mount
Drive, restore data tars.

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/phanindra-max/Cross-Generator-Generalization-Project.git"
REPO_NAME = "Cross-Generator-Generalization-Project"

if not Path("src").exists() and not Path(REPO_NAME).exists():
    !git clone -q {REPO_URL}

if Path(REPO_NAME, "src").exists() and not Path("src").exists():
    %cd {REPO_NAME}

!pip install -q uv
!uv pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
assert (PROJECT_ROOT / "src").exists(), f"src/ not found under {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working dir: {PROJECT_ROOT}")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BACKUP = Path("/content/drive/MyDrive/deepfake-cross-generator/backups")
    IN_COLAB = True
    print(f"Drive backup root: {DRIVE_BACKUP}")
except ImportError:
    IN_COLAB = False
    DRIVE_BACKUP = None
    print("Not in Colab — Drive mount skipped.")

In [ ]:
# Restore original (cv2-rendered) data from Drive so we have something to
# compare against. data_unified/ is freshly built by this notebook so we
# never restore it from Drive on the first run.
import subprocess

ARTIFACTS = [
    ("data/real",           "real.tar"),
    ("data/faceswap",       "faceswap.tar"),
    ("data/diffusion_sd15", "diffusion_sd15.tar"),
]

def _is_empty(p: Path) -> bool:
    if not p.exists():
        return True
    return not any(not c.name.startswith(".") for c in p.iterdir())

if IN_COLAB and DRIVE_BACKUP and DRIVE_BACKUP.exists():
    for local, archive in ARTIFACTS:
        target = Path(local)
        src = DRIVE_BACKUP / archive
        if not _is_empty(target):
            print(f"  skip {local}: already populated")
            continue
        if not src.exists():
            print(f"  miss {local}: no archive in Drive yet")
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-xf", str(src), "-C", str(target.parent)], check=True)
        print(f"  ok   {local} restored from {archive}")
else:
    print("No Drive backup root — skipping restore.")

for local, _ in ARTIFACTS:
    p = Path(local)
    n = len(list(p.glob("*.png"))) if p.exists() else 0
    print(f"  {local:<24} {n} PNGs")

## Pre-flight A — bit-identity check

Decode 20 random `data/faceswap/*.png` files via both cv2 and PIL,
report max absolute pixel difference per file. If every file shows 0,
the encoder split (cv2.imwrite vs PIL.save) is provably not the leak
and we don't need to test it further.

In [ ]:
import random
from src.data.diagnostic_pipeline import bit_identity_check

random.seed(0)
candidates = sorted(Path("data/faceswap").glob("*.png"))
sample = random.sample(candidates, k=min(20, len(candidates)))
results = bit_identity_check(str(p) for p in sample)

max_diffs = [d for _, d in results]
print(f"Files checked: {len(results)}")
print(f"max_abs_diff distribution: min={min(max_diffs)}, max={max(max_diffs)}")
for path, diff in results[:5]:
    print(f"  {Path(path).name}: max_abs_diff = {diff}")
if max(max_diffs) == 0:
    print("\nAll decoders agree to the bit. The encoder split (cv2 vs PIL) cannot leak.")
else:
    print(f"\nNon-zero diff found (max={max(max_diffs)}). Worth investigating.")

## Pre-flight B — resize-kernel demo

Take one high-res source frame from the Kaggle FaceSwap directory,
downsample it to 128 via both `cv2.INTER_AREA` and `PIL.LANCZOS`, and
report the pixel diff. A non-trivial max diff confirms the two
kernels produce different outputs at the pixel level — which is what a
high-pass front end would amplify. If the Kaggle source isn't
available locally, this cell will print a note and skip.

In [ ]:
from src.data.diagnostic_pipeline import demo_resize_kernel_diff
from src.data.prepare_faceforensics import (
    download_faceforensics_kaggle, find_faceswap_dir,
    _list_image_frames, _list_video_files,
)

try:
    kaggle_root = download_faceforensics_kaggle()
    faceswap_dir = find_faceswap_dir(kaggle_root)
    print(f"FaceSwap source: {faceswap_dir}")

    # Find one source frame: prefer a direct image, then descend into the
    # first sub-dir, then fall back to extracting frame 0 from a video.
    sample_path = None
    direct = _list_image_frames(faceswap_dir)
    if direct:
        sample_path = direct[0]
    else:
        for child in sorted(faceswap_dir.iterdir()):
            if child.is_dir():
                imgs = _list_image_frames(child)
                if imgs:
                    sample_path = imgs[0]
                    break
        if sample_path is None:
            videos = _list_video_files(faceswap_dir)
            if videos:
                import cv2
                cap = cv2.VideoCapture(str(videos[0]))
                ok, frame = cap.read()
                cap.release()
                if ok:
                    sample_path = videos[0].with_suffix(".extracted.png")
                    cv2.imwrite(str(sample_path), frame)

    if sample_path is None:
        print("No source frame located — skipping demo.")
    else:
        out = demo_resize_kernel_diff(str(sample_path), resolution=128)
        print(f"  source size:           {out['src_size']}")
        print(f"  max abs diff (uint8):  {out['max_abs_diff_uint8']}")
        print(f"  mean abs diff (uint8): {out['mean_abs_diff_uint8']:.3f}")
        print(f"  cv2.INTER_AREA  output: {out['cv2_path']}")
        print(f"  PIL.LANCZOS     output: {out['pil_path']}")
        if out['max_abs_diff_uint8'] > 0:
            print("\nKernels produce visibly different pixels. The high-pass front end will see this.")
except Exception as exc:
    print(f"Kaggle source not available locally ({exc!r}); skipping demo.")

## Step C — re-render FaceSwap through a PIL pipeline

Same source set as the original prep (same seed, same `num_videos`,
same `frames_per_video`) so each output frame corresponds 1:1 to a
frame in the original `data/faceswap/`. Only the resize kernel and
encoder differ.

In [ ]:
from src.data.diagnostic_pipeline import rerender_faceswap_pil

n_written = rerender_faceswap_pil(
    output_dir="data_unified/faceswap",
    num_videos=500,
    frames_per_video=4,
    resolution=128,
    seed=42,
)
print(f"Total: {n_written} frames at data_unified/faceswap/")

In [ ]:
# Visual-ish sanity check: pick 3 matching filenames between the cv2 and
# PIL renders and report pixel-level differences. Should be non-zero (the
# resamplers really differ) but not catastrophic (same source frames).
import numpy as np
from PIL import Image

cv2_dir = Path("data/faceswap")
pil_dir = Path("data_unified/faceswap")
common = sorted(set(p.name for p in cv2_dir.glob("*.png")) & set(p.name for p in pil_dir.glob("*.png")))
if not common:
    print("No matching filenames — source-set or seed contract differs.")
else:
    for name in common[:3]:
        a = np.array(Image.open(cv2_dir / name).convert("RGB"), dtype=np.int32)
        b = np.array(Image.open(pil_dir / name).convert("RGB"), dtype=np.int32)
        diff = np.abs(a - b)
        print(f"  {name}: max={diff.max():3d}  mean={diff.mean():.3f}")

## Step D — regenerate splits pointing at the unified FaceSwap

Single-variable change: `data/real/` (PIL, no resize) and
`data/diffusion_sd15/` (PIL.LANCZOS) are reused as-is. Only the
FaceSwap path is swapped. Output goes to `data_unified/splits/`.

In [ ]:
from src.data.create_splits import create_splits

create_splits(
    real_dir="data/real",
    faceswap_dir="data_unified/faceswap",
    diffusion_dir="data/diffusion_sd15",
    output_dir="data_unified/splits",
    seed=42,
)

## Step E — back up unified data + splits to Drive

Same atomic tar-then-rename pattern as notebook 00. Idempotent.

In [ ]:
if IN_COLAB and DRIVE_BACKUP is not None:
    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    for local, archive in [
        ("data_unified/faceswap", "faceswap_unified.tar"),
        ("data_unified/splits",   "splits_unified.tar"),
    ]:
        src = Path(local)
        if not src.exists() or not any(src.iterdir()):
            print(f"  skip {local}: empty")
            continue
        final = DRIVE_BACKUP / archive
        tmp = final.with_suffix(final.suffix + ".tmp")
        subprocess.run(["tar", "-cf", str(tmp), "-C", str(src.parent), src.name], check=True)
        tmp.replace(final)
        size_mb = final.stat().st_size / (1024 * 1024)
        print(f"  ok   {local} -> {archive} ({size_mb:.1f} MB)")
else:
    print("Not in Colab or no Drive — skipping backup.")

## What's next — retrain instructions

In `02_train_models.ipynb`, find the line in section 2.1:

```python
SPLITS_DIR     = Path("data/splits")
```

and change it to:

```python
SPLITS_DIR     = Path("data_unified/splits")
```

Optionally rename the checkpoint dir too so you don't overwrite the
original-pipeline checkpoints — e.g. `results/checkpoints_unified/`.
Then re-run cells 2.5 and 2.6 to retrain ConstrainedCNN and
BaselineResNet against the unified data.

Outcome map:
- ConstrainedCNN in-dist AUC drops meaningfully (e.g. 1.000 → < 0.95)
  and OOD AUC moves → the resize-kernel asymmetry was the leak.
- ConstrainedCNN in-dist AUC stays ~1.000 → the model is finding
  something more durable (JPEG ghosts from the c23 source, FaceSwap
  blending artifacts, etc.). Different and more interesting story for
  the writeup.
- BaselineResNet numbers stay roughly the same on both → strengthens
  the "naturalness prior" interpretation, since ResNet wasn't relying
  on the resampler cue in the first place.

Then run `03_cross_generator_eval.ipynb` to compare cross-generator
transfer numbers between the original-pipeline and unified-pipeline
checkpoints.